# Task 1: Sales & Demand Forecasting

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Set plot style
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Data Generation
Since we don't have a dataset, we will generate synthetic daily sales data with trend and seasonality.

In [ ]:
# Generate date range
dates = pd.date_range(start="2023-01-01", end="2024-12-31", freq="D")
n = len(dates)

# specific random seed for reproducibility
np.random.seed(42)

# Create components
trend = np.linspace(100, 300, n)
seasonality = 50 * np.sin(2 * np.pi * np.arange(n) / 365) + 20 * np.cos(2 * np.pi * np.arange(n) / 90)
noise = np.random.normal(0, 15, n)

# Combine to form sales
sales = trend + seasonality + noise
sales = np.maximum(sales, 0)  # Ensure no negative sales

# Create DataFrame
df = pd.DataFrame({"Date": dates, "Sales": sales})
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(df["Date"], df["Sales"], label="Daily Sales", color='blue', alpha=0.7)
plt.title("Historical Sales Data")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.show()

## 4. Feature Engineering
Extract features from the date to use in our regression model.

In [ ]:
df['Day'] = df['Date'].dt.day
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Quarter'] = df['Date'].dt.quarter

# Determine trend feature (numeric representation of time)
df['TimeIndex'] = np.arange(len(df))

df.head()

## 5. Model Training

In [ ]:
# Define features and target
features = ['Day', 'Month', 'Year', 'DayOfWeek', 'Quarter', 'TimeIndex']
target = 'Sales'

X = df[features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Initialize and train model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

print("Model Trained Successfully")

## 6. Evaluation

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

## 7. Visualization of Results

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(df['Date'], df['Sales'], label='Actual Sales', color='blue', alpha=0.5)
plt.plot(df['Date'].iloc[len(X_train):], y_pred, label='Predicted Sales', color='red', linestyle='--')
plt.axvline(df['Date'].iloc[len(X_train)], color='black', linestyle=':', label='Train/Test Split')
plt.title("Sales Forecasting: Actual vs Predicted")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.show()

## 8. Forecast Future Sales

In [ ]:
# Generate future dates
future_dates = pd.date_range(start="2025-01-01", end="2025-01-31", freq="D")
future_df = pd.DataFrame({"Date": future_dates})

# Create features for future dates
future_df['Day'] = future_df['Date'].dt.day
future_df['Month'] = future_df['Date'].dt.month
future_df['Year'] = future_df['Date'].dt.year
future_df['DayOfWeek'] = future_df['Date'].dt.dayofweek
future_df['Quarter'] = future_df['Date'].dt.quarter
future_df['TimeIndex'] = np.arange(len(df), len(df) + len(future_df))

# Predict
future_predictions = model.predict(future_df[features])

# Visualize Future Forecast
plt.figure(figsize=(10, 5))
plt.plot(future_dates, future_predictions, marker='o', linestyle='-', color='green', label='Future Forecast')
plt.title("Sales Forecast for Jan 2025")
plt.xlabel("Date")
plt.ylabel("Predicted Sales")
plt.grid(True)
plt.legend()
plt.show()